[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-3-ai-agents/adv-01b-completion-checks-and-pipeline-vs-agent.ipynb)

# Advanced Discussion 1 (continued) — completion gates and the pipeline-versus-agent bill

A simulation of an agent loop with and without an external completion check, and the token arithmetic of a pipeline versus an agent. Prices are placeholders.

**Optional advanced-discussion lab** for AI Agents and Agentic AI with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Simulation only; runs in seconds.

In [ ]:
%pip install -q numpy

In [ ]:
import numpy as np, math
rng = np.random.RandomState(0)

## 1. "done" is a claim, not a fact: an agent loop with and without an external completion check

In [ ]:
def run(gate, recall=0.95, false_alarm=0.03, miss_self=0.5, fix_uninformed=0.3, fix_informed=0.85, max_iter=12, max_rejects=3, seed=0):
    r = np.random.RandomState(seed); E = r.poisson(1.5); informed = False; it = 0; calls = 0; rejects = 0; claimed_done_wrong = False
    while it < max_iter:
        it += 1
        if E > 0 and r.rand() < (fix_informed if informed else fix_uninformed): E -= 1                     # a productive iteration
        claims_done = (E == 0) or (r.rand() < miss_self)                                                   # the agent's own check misses remaining errors half the time
        if not claims_done: continue
        if not gate:
            return E == 0, it, 0, (E > 0)
        calls += 1                                                                                         # run the external check (tests, validator, cross-check)
        if E > 0:
            if r.rand() < recall:
                rejects += 1; informed = True
                if rejects > max_rejects: return E == 0, it, calls, False
                continue
            return False, it, calls, True                                                                  # verifier missed it: accepted a wrong result
        else:
            if r.rand() < false_alarm: rejects += 1; continue                                              # spurious rejection: a wasted iteration
            return True, it, calls, False
    return E == 0, it, calls, False
def stats(**kw):
    res = [run(seed=s, **kw) for s in range(20000)]; ok = np.mean([x[0] for x in res]); its = np.mean([x[1] for x in res]); vc = np.mean([x[2] for x in res]); wrong_accept = np.mean([x[3] for x in res])
    return ok, its, vc, wrong_accept
print("task starts with Poisson(1.5) errors to fix; the agent's own check misses remaining errors 50% of the time; at most 12 iterations")
print("configuration                                      correct   iterations   verifier calls   wrong result accepted")
for name, kw in (("trust the agent's 'done'", dict(gate=False)), ("gate: verifier recall 0.95, false alarm 3%", dict(gate=True)),
                 ("gate: weak verifier, recall 0.60", dict(gate=True, recall=0.60)), ("gate: noisy verifier, false alarm 20%", dict(gate=True, false_alarm=0.20)),
                 ("gate: perfect verifier", dict(gate=True, recall=1.0, false_alarm=0.0))):
    ok, its, vc, wa = stats(**kw); print("%-50s %.3f     %5.2f        %5.2f            %.3f" % (name, ok, its, vc, wa))
print("\nverifier recall sweep (false alarm 3%):")
for rc in (0.0, 0.3, 0.6, 0.8, 0.95, 1.0): ok, its, vc, wa = stats(gate=True, recall=rc); print("  recall %.2f: correct %.3f, iterations %.2f" % (rc, ok, its))

## 2. pipeline vs agent for a task whose steps are known: extract 4 fields from a 2,000-token document

In [ ]:
IN, OUT = 3.00, 15.00                                               # PLACEHOLDER $/M tokens
DOC, PROMPT = 2000, 300
def cost_tokens(inp, out): return (inp * IN + out * OUT) / 1e6
# pipeline: one call reads the document and returns all four fields as JSON
pipe_in, pipe_out = DOC + PROMPT, 90
# agent: plan, then a tool call per field that fetches a ~500-token passage, then a final answer. Every call re-sends the whole history.
def agent_cost(fields=4, passage=500, step_out=60):
    hist = PROMPT; total_in = 0; total_out = 0; calls = 0
    for _ in range(fields + 2):                                     # plan + one lookup per field + answer
        total_in += hist; total_out += step_out; calls += 1; hist += step_out + passage
    return total_in, total_out, calls
ai, ao, ac = agent_cost()
print("\nextracting 4 fields from a %d-token document (placeholder prices):" % DOC)
print("  pipeline: 1 model call,  %5d input + %3d output tokens = $%.4f" % (pipe_in, pipe_out, cost_tokens(pipe_in, pipe_out)))
print("  agent:    %d model calls, %5d input + %3d output tokens = $%.4f  (%.1fx the pipeline)" % (ac, ai, ao, cost_tokens(ai, ao), cost_tokens(ai, ao) / cost_tokens(pipe_in, pipe_out)))
# reliability under stated per-step assumptions (ASSUMED numbers - the point is how they combine, not the values)
p_pipe = 0.97 ** 1 * 0.99                                            # one reading step + a schema check that catches malformed output
p_agent = 0.97 ** (4 + 2)
print("  success under assumed per-step reliability 0.97 (agent needs %d successful steps; pipeline 1 step and a validator): pipeline %.3f vs agent %.3f" % (6, p_pipe, p_agent))
# hybrid: pipeline does the four known steps, and a bounded agent is used only when validation fails (assume 8% of documents)
frac_hard = 0.08; hard_cost = cost_tokens(*agent_cost()[:2]); hybrid = cost_tokens(pipe_in, pipe_out) + frac_hard * hard_cost
print("  hybrid: pipeline first, bounded agent only for the %d%% that fail validation: $%.4f per document (%.1fx the pipeline)" % (100 * frac_hard, hybrid, hybrid / cost_tokens(pipe_in, pipe_out)))
print("  for 100,000 documents: pipeline $%.0f | agent $%.0f | hybrid $%.0f" % (1e5 * cost_tokens(pipe_in, pipe_out), 1e5 * cost_tokens(ai, ao), 1e5 * hybrid))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.